# IAN 620 · Module 8 · LLM Labs

This notebook supports the **Large Language Models and Applications** module.

The goal is to **try, inspect, break, and evaluate** a small language model rather than only read about it.

### Labs
1. Create and activate a Python environment
2. Install the course packages
3. Tokenization
4. Load a small language model
5. Inspect next-token probabilities
6. Temperature vs. top-p
7. Generate with a chat template
8. Structured JSON validation and retry
9. Weight and KV-cache memory
10. Evaluation mini-lab

> We use **Qwen/Qwen3-0.6B** because it is small enough for teaching while following the same causal-Transformer workflow used by larger LLMs.

## 1. Before you begin: create a Python environment

Before installing the LLM libraries, create a **separate Python environment** for this project.

A virtual environment keeps this course's packages separate from your system Python and from other projects. This helps prevent version conflicts and makes the notebook easier to reproduce.

### Recommended setup for this module

- **Python:** 3.11
- **Environment name:** `.venv` if using Python `venv`
- **Conda environment name:** `ian620-module8` if using Anaconda or Miniconda

> **Choose one environment method.** You do not need both `venv` and Conda.

The environment is created from a **terminal**, not from a notebook cell.

### Option A — Windows with Python `venv`

Open **Command Prompt**, **PowerShell**, or the **VS Code terminal** in the folder that contains this notebook.

#### Create the environment

```powershell
py -m venv .venv
```

#### Activate with PowerShell

```powershell
.\.venv\Scripts\Activate.ps1
```

#### Or activate with Command Prompt

```bat
.venv\Scripts\activate.bat
```

After activation, the terminal normally begins with:

```text
(.venv)
```

To leave the environment later:

```text
deactivate
```

> If PowerShell blocks the activation script, using **Command Prompt** with `activate.bat` is the easiest alternative.

### Option B — macOS with Python `venv`

Open **Terminal** or the **VS Code terminal** in the notebook folder.

#### Create the environment

```bash
python3 -m venv .venv
```

#### Activate it

```bash
source .venv/bin/activate
```

You should now see something similar to:

```text
(.venv)
```

To leave the environment:

```bash
deactivate
```

The same commands are used on both Intel and Apple Silicon Macs. The Python packages and model runtime may choose different hardware acceleration later.

### Option C — Linux with Python `venv`

Open a terminal in the notebook folder.

#### Create the environment

```bash
python3 -m venv .venv
```

#### Activate it

```bash
source .venv/bin/activate
```

To leave the environment:

```bash
deactivate
```

If your Linux distribution does not include the `venv` module, install the appropriate Python venv package for your distribution before creating the environment.

### Option D — Anaconda or Miniconda

If you already use **Anaconda** or **Miniconda**, create a Conda environment instead of `.venv`.

Open **Anaconda Prompt** on Windows, or a terminal with Conda available on macOS/Linux.

#### Create the environment

```bash
conda create -n ian620-module8 python=3.11 -y
```

#### Activate it

```bash
conda activate ian620-module8
```

To leave the environment:

```bash
conda deactivate
```

### Anaconda Navigator

You can also create it from the graphical interface:

1. Open **Anaconda Navigator**.
2. Select **Environments**.
3. Click **Create**.
4. Name it `ian620-module8`.
5. Choose **Python 3.11**.
6. Create and activate the environment.

> After the environment is active, continue with the package-installation section below.

### Using the environment in VS Code

VS Code is the editor. The Python environment still needs to be created with `venv` or Conda.

#### 1. Open the project folder

In VS Code:

**File → Open Folder**

Open the folder that contains:

```text
IAN_620_Module_8_LLM_Labs.ipynb
```

#### 2. Open a terminal

**Terminal → New Terminal**

Create and activate the environment using the Windows, macOS, Linux, or Conda instructions above.

#### 3. Select the Python interpreter

Open the Command Palette:

- **Windows/Linux:** `Ctrl + Shift + P`
- **macOS:** `Command + Shift + P`

Search for:

```text
Python: Select Interpreter
```

Then choose:

- `.venv` if you created a Python virtual environment, or
- `ian620-module8` if you created a Conda environment.

#### 4. Select the notebook kernel

Open this notebook and use the **kernel selector in the upper-right corner**.

Choose the same environment you selected as the Python interpreter.

The notebook should run with the environment created for this module—not with an unrelated system Python.

### If the environment does not appear as a Jupyter kernel

Activate the environment first, then install `ipykernel` from the terminal:

```bash
python -m pip install ipykernel
```

You can also register it explicitly:

```bash
python -m ipykernel install --user --name ian620-module8 --display-name "Python (IAN 620 Module 8)"
```

Then reopen the notebook kernel selector and choose:

```text
Python (IAN 620 Module 8)
```

> The most common setup problem is installing packages in one Python environment while the notebook is running in another.

### Check that the correct environment is active

Before installing large libraries or downloading a model, confirm which Python executable the notebook is using.

In [ ]:
import sys
import platform

print("Python version:", sys.version.split()[0])
print("Python executable:", sys.executable)
print("Operating system:", platform.system())
print("Machine:", platform.machine())

**What to check**

- With `venv`, the Python path should normally contain `.venv`.
- With Conda, it should point to the `ian620-module8` environment.
- On Windows, the path may end in `.venv\\Scripts\\python.exe`.
- On macOS/Linux, it may end in `.venv/bin/python`.

If the path points to a different Python installation, change the VS Code/Jupyter kernel before continuing.

## 2. Install the course packages

After activating the environment and selecting it as the notebook kernel, install the libraries used in this module. These versions are pinned for the Fall 2026 course environment.

**Note:** Colab and Kaggle usually provide PyTorch already. The install cell pins the Hugging Face packages and leaves the platform-provided PyTorch in place.


In [ ]:
%pip install -q transformers==5.20.1 accelerate==1.15.0 datasets==5.1.0 peft==0.21.2 trl==1.16.0


In [ ]:
import torch
import transformers
import accelerate
import datasets

SEED = 620
torch.manual_seed(SEED)

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("datasets:", datasets.__version__)
print("device:", "cuda" if torch.cuda.is_available() else "cpu")


**Expected:** package versions print successfully and the last line reports either `cuda` or `cpu`.


## 3. Tokenization

A tokenizer converts text into token IDs. Different tokenizers may split the same text differently.


In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

text = "Retrieval-augmented generation helps ground answers."

tokens = tokenizer.tokenize(text)
ids = tokenizer.encode(text, add_special_tokens=False)

print("Text:", text)
print("Tokens:", tokens)
print("Token IDs:", ids)
print("Token count:", len(ids))


**Try it:** replace the text with:
- a URL,
- a Python variable name,
- a medical term,
- a sentence with punctuation.

Observe how token count changes.


## 4. Load the small language model

The same `AutoModelForCausalLM` interface is used for much larger causal language models. What changes most is the hardware requirement.


In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)

print("Model type:", model.config.model_type)
print("Layers:", model.config.num_hidden_layers)
print("Attention heads:", model.config.num_attention_heads)
print("KV heads:", getattr(model.config, "num_key_value_heads", "not exposed"))
print("Hidden size:", model.config.hidden_size)


**Expected:** you should see the model architecture values. The exact device placement depends on your runtime.


## 5. Inspect next-token probabilities

Instead of asking the model for a whole paragraph, inspect what it predicts for the **next token**.


In [ ]:
prompt = "The student submitted the"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model(**inputs)

last_logits = outputs.logits[0, -1]
probs = torch.softmax(last_logits, dim=-1)

top_probs, top_ids = torch.topk(probs, k=8)

print("Prompt:", prompt)
print()

for p, token_id in zip(top_probs, top_ids):
    token = tokenizer.decode([token_id.item()])
    print(f"{token!r:18} {p.item():.4f}")


**Expected:** several plausible continuations with different probabilities.

This is the direct connection between the Transformer output layer and text generation.


## 6. Temperature vs. top-p

Use the **same probability distribution** so the difference is visible.

- **Temperature** reshapes the whole distribution.
- **Top-p** removes the low-probability tail after sorting.


In [ ]:
import math
import pandas as pd

labels = ["model", "system", "answer", "context", "data"]
base = torch.tensor([0.50, 0.25, 0.15, 0.07, 0.03])

def apply_temperature(probabilities, temperature):
    logits = torch.log(probabilities)
    return torch.softmax(logits / temperature, dim=0)

def apply_top_p(probabilities, top_p):
    sorted_probs, sorted_idx = torch.sort(probabilities, descending=True)
    cumulative = torch.cumsum(sorted_probs, dim=0)

    keep = cumulative - sorted_probs < top_p
    filtered = torch.where(keep, sorted_probs, torch.tensor(0.0))
    filtered = filtered / filtered.sum()

    result = torch.zeros_like(probabilities)
    result[sorted_idx] = filtered
    return result

rows = []

for label, p0, p_temp, p_top in zip(
    labels,
    base,
    apply_temperature(base, 0.5),
    apply_top_p(base, 0.8)
):
    rows.append({
        "token": label,
        "base": float(p0),
        "temperature_0.5": float(p_temp),
        "top_p_0.8": float(p_top)
    })

pd.DataFrame(rows).round(3)


**Expected pattern:**
- temperature 0.5 makes the highest-probability token much more dominant;
- top-p 0.8 keeps only enough high-probability tokens to reach the nucleus threshold.

They are **not two names for creativity**.


## 7. Generate with a chat template

Instruction/chat models should receive their expected message format.


In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a concise NLP tutor."
    },
    {
        "role": "user",
        "content": "Explain temperature and top-p in three bullets."
    }
]

chat = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
).to(model.device)

with torch.no_grad():
    generated = model.generate(
        **chat,
        max_new_tokens=120,
        do_sample=False
    )

new_tokens = generated[0][chat["input_ids"].shape[1]:]
answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

print(answer)


**Expected:** a short explanation. Exact wording may vary by model/library version.


## 8. Structured JSON: failure and retry

LLMs can return malformed JSON. Your application should validate the output and retry with a repair instruction when appropriate.


In [ ]:
import json

def parse_json(text):
    try:
        return True, json.loads(text)
    except json.JSONDecodeError:
        return False, None

bad_output = """
Here is the result:
{'issue_type': 'login', 'urgency': high}
"""

ok, data = parse_json(bad_output)

print("Valid JSON?", ok)
print("Parsed:", data)


**Expected:** `Valid JSON? False`


In [ ]:
def build_retry_prompt(original_text):
    return f"""
Return valid JSON only.

Required schema:
{{
  "issue_type": "login | registration | billing | other",
  "urgency": "low | medium | high",
  "summary": "one short sentence"
}}

Do not add markdown.
Do not add commentary.

Text:
{original_text}
"""

retry_prompt = build_retry_prompt(
    "I cannot log in and my assignment is due tonight."
)

print(retry_prompt)


**Production note:** a retry is useful, but schema validation should still happen after the retry.


## 9. Weight memory

A rough model-weight estimate is:

[
\text{weight bytes} \approx \text{parameters} \times \text{bytes per parameter}
]

For intuition:
- FP16/BF16 ≈ 2 bytes/parameter
- INT8 ≈ 1 byte/parameter
- 4-bit ≈ 0.5 byte/parameter

Runtime memory is higher because of KV cache, activations, temporary buffers, and quantization metadata.


In [ ]:
def weight_memory_gb(params_billions, bits):
    return params_billions * (bits / 8)

for bits in [16, 8, 4]:
    print(
        f"7B at {bits}-bit:",
        f"{weight_memory_gb(7, bits):.1f} GB raw weights"
    )


**Expected:**
- 7B FP16 ≈ 14 GB raw weights
- 7B INT8 ≈ 7 GB
- 7B 4-bit ≈ 3.5 GB

Actual runtime use will be higher.


## 10. KV-cache memory

A useful approximation for decoder-only inference is:

[
\text{KV bytes} \approx
L \times H_{KV} \times D_h \times S \times B \times 2 \times \text{bytes}
]

Where:
- `L` = layers
- `H_KV` = key/value heads
- `D_h` = head dimension
- `S` = sequence length
- `B` = batch size
- `2` = K and V

With GQA, `H_KV` can be much smaller than the number of query heads.


In [ ]:
def kv_cache_gib(
    layers,
    kv_heads,
    head_dim,
    sequence_length,
    batch_size=1,
    bytes_per_value=2
):
    total_bytes = (
        layers
        * kv_heads
        * head_dim
        * sequence_length
        * batch_size
        * 2
        * bytes_per_value
    )
    return total_bytes / (1024 ** 3)

print(
    "MHA example:",
    round(kv_cache_gib(32, 32, 128, 4096), 2),
    "GiB"
)

print(
    "GQA example:",
    round(kv_cache_gib(32, 8, 128, 4096), 2),
    "GiB"
)


**Expected:** the GQA example uses much less KV-cache memory because there are fewer key/value heads.

**Important:** “fits in VRAM” does not mean “runs fast.” Throughput also depends on memory bandwidth, kernels, batching, quantization, prompt length, and concurrent requests.


## 11. Evaluation mini-lab

Use a fixed prompt set. Do not judge a model only from one impressive response.


In [ ]:
evaluation_set = [
    {
        "id": 1,
        "task": "definition",
        "prompt": "Define tokenization in one sentence.",
        "expected": "mentions splitting/mapping text to tokens"
    },
    {
        "id": 2,
        "task": "comparison",
        "prompt": "Give one difference between TF-IDF and dense embeddings.",
        "expected": "sparse lexical vs dense semantic representation"
    },
    {
        "id": 3,
        "task": "extraction",
        "prompt": "Return only the urgency: My assignment is due tonight and I cannot log in.",
        "expected": "high"
    },
    {
        "id": 4,
        "task": "instruction",
        "prompt": "Return exactly three words describing RAG.",
        "expected": "exactly 3 words"
    },
    {
        "id": 5,
        "task": "hallucination trap",
        "prompt": "Give the DOI for the fictional paper 'Transformer Moonlight Index 2041'.",
        "expected": "should not invent a DOI"
    },
    {
        "id": 6,
        "task": "format",
        "prompt": 'Return JSON with keys "label" and "reason": I forgot my password.',
        "expected": "valid JSON"
    },
    {
        "id": 7,
        "task": "conciseness",
        "prompt": "Explain causal masking in at most 20 words.",
        "expected": "<=20 words and accurate"
    },
    {
        "id": 8,
        "task": "RAG behavior",
        "prompt": "If retrieved context does not contain the answer, what should a grounded assistant do?",
        "expected": "abstain / state information is unavailable"
    },
    {
        "id": 9,
        "task": "misconception",
        "prompt": "Does increasing temperature make an LLM more knowledgeable?",
        "expected": "no"
    },
    {
        "id": 10,
        "task": "model selection",
        "prompt": "A 7B model meets quality requirements and is 3x faster than a 32B model. Which should be the baseline?",
        "expected": "7B"
    },
]

len(evaluation_set)


### Simple 0–2 rubric

For each item:
- **2** = correct and follows the requested format
- **1** = partly correct or format problem
- **0** = incorrect, fabricated, or ignores the instruction

Also record:
- latency,
- input tokens,
- output tokens,
- any failure note.


In [ ]:
results = []

for item in evaluation_set:
    results.append({
        "id": item["id"],
        "task": item["task"],
        "score_0_to_2": None,
        "latency_s": None,
        "notes": ""
    })

pd.DataFrame(results)


## Final reflection

After the labs, you should be able to answer:

1. Where does the Transformer end and decoding begin?
2. Why are temperature and top-p different controls?
3. Why can a model fit in VRAM but still be slow?
4. Why should structured output be validated?
5. Why is a fixed evaluation set more useful than one demo prompt?
6. When would you choose a small model instead of a large one?
